# 04 - Modeling

This notebook trains a **baseline model** (to provide a reference accuracy score to compare more advanced models) and a **logistic regression model**, evaluates both, and interprets the logistic regression coefficients as odds ratios.

Training and evaluation logic lives in `src/models/train.py` and `src/models/evaluate.py` - this notebook calls those functions.

In [1]:
import sys, os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
# print("Working directory:", Path.cwd())

In [2]:
from src.data.load_data import load_config, load_data
from src.data.preprocess import clean_data
from src.features.feature_engineering import (
    split_features_target,
    split_train_test,
    scale_features,
)
from src.models.train import train_baseline, train_logistic_regression, save_artifacts
from src.models.evaluate import evaluate_model, coefficients_to_odds_ratios

## Prepare data

In [3]:
config = load_config()
df = load_data(config)
df_clean = clean_data(df, config["data"]["invalid_zero_columns"])
X, y = split_features_target(df_clean, config["data"]["target_column"])
X_train, X_test, y_train, y_test = split_train_test(X, y, config)
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
print("Ready:", X_train_scaled.shape, X_test_scaled.shape)

Ready: (614, 8) (154, 8)


## Baseline model

A majority-class `DummyClassifier` - predicts the most frequent class every time. This sets the bar the real model needs to clear to be considered useful at all.

In [4]:
baseline = train_baseline(X_train_scaled, y_train)
baseline_results = evaluate_model(baseline, X_test_scaled, y_test, predict_proba=False)
print("Baseline accuracy:", baseline_results["accuracy"])
print()
print(f"Baseline Classification Report: \n{baseline_results["classification_report"]}")

Baseline accuracy: 0.6493506493506493

Baseline Classification Report: 
              precision    recall  f1-score   support

           0       0.65      1.00      0.79       100
           1       0.00      0.00      0.00        54

    accuracy                           0.65       154
   macro avg       0.32      0.50      0.39       154
weighted avg       0.42      0.65      0.51       154



### Baseline Model Evaluation  

Before building predictive models, we establish a baseline using a simple **majority-class classifier**, which always predicts the most frequent class (`Outcome = 0`, non-diabetic). This provides a reference point to assess whether our predictive models add value.

#### Baseline Model Results
- **Accuracy:** 65%

    - Matches the proportion of non-diabetic participants in the dataset (~65%).
    - Shows that a naïve model can achieve reasonable accuracy by always predicting the majority class.

- Class-specific performance:

    - **Non-diabetic (0):** Precision ≈ 0.65, Recall = 1.0, F1-score ≈ 0.79
        - The model correctly identifies all non-diabetic cases because it always predicts 0.
    - **Diabetic (1):** Precision = 0.0, Recall = 0.0, F1-score = 0.0
        - The model fails to predict any diabetic cases, which highlights the need for a real model that can identify the minority class.

#### Key Takeaways
- The baseline model provides a **minimum performance benchmark.**
- Any meaningful predictive model must:
    1. Correctly identify some diabetic cases.
    2. Improve metrics for the minority class (recall, F1-score, ROC-AUC) while maintaining overall accuracy.

## Logistic regression

The real model: `LogisticRegression(max_iter=1000, random_state=42)`, trained on the scaled features.  

**Logistic Regression** is a statistical model suitable for binary classification tasks, such as predicting whether a patient is diabetic or not.

Key reasons for using Logistic Regression:

- It outputs probabilities, allowing flexible decision thresholds.
- Model coefficients provide interpretability, indicating the direction and magnitude of influence of each feature.
- It performs well on linearly separable data and serves as a strong baseline before trying more complex models.  

We will:

1. Train the model using the standardized training set.
2. Make predictions on the test set.
3. Evaluate performance using:
    - Accuracy
    - Precision, Recall, F1-score
    - Confusion Matrix
    - ROC-AUC Score

In [5]:
log_reg = train_logistic_regression(X_train_scaled, y_train, config)
lr_results = evaluate_model(log_reg, X_test_scaled, y_test)
print("Accuracy:", lr_results["accuracy"])
print("ROC-AUC: ", lr_results["roc_auc"])

Accuracy: 0.7077922077922078
ROC-AUC:  0.812962962962963


In [6]:
print("Logistic Regression Classification Report:")
print(lr_results["classification_report"])

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.82      0.78       100
           1       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154



In [7]:
print("Confusion matrix:")
print(lr_results["confusion_matrix"])

Confusion matrix:
[[82 18]
 [27 27]]


### Logistic Regression Model Results & Interpretation

After training the Logistic Regression classifier model, we evaluate its performance on the test set. Below is the summary of the model's performance metrics.

#### Performance Summary

- **Accuracy:** ~70.8% - this is an improvement over the baseline accuracy of ~65%.
- **ROC-AUC Score:** ~0.813 - indicates good discrimination capability between diabetic and non-diabetic cases.

#### Classification Performance by Class

| Metric | Class 0 (Non-diabetic) | Class 1 (Diabetic) |
|---|---:|---:|
| Precision | 0.75 | 0.60 |
| Recall | 0.82 | 0.50 |
| F1-score | 0.78 | 0.55 |

#### Interpretation

- The model performs better for **non-diabetic (majority class)**, correctly identifying 82 out of 100 patients.
- For diabetic (minority class), recall increases from **0.0 (baseline)** to **0.50**, meaning the model now correctly identifies **half of the diabetic cases**, a significant improvement.
- This aligns with typical performance challenges in **imbalanced medical datasets**, where identifying positive cases is harder.

#### Confusion Matrix

|  | Predicted 0 | Predicted 1 |
|---|---:|---:|
| **Actual 0** | 82 | 18 |
| **Actual 1** | 27 | 27 |

- **False negatives (27):** this remains an important concern since misclassifying diabetic patients can have clinical consequences.
- A possible future step may involve **adjusting decision thresholds**, **using class weights**, or **trying more advanced models** to improve recall for the positive class.

#### Model Assessment

Compared to the baseline model (which predicted only the majority class), Logistic Regression:

- Correctly identifies diabetic patients.
- Achieves higher performance across all key metrics.
- Shows promising predictive power with an AUC above 0.80.

Overall, Logistic Regression provides a **meaningful improvement** and serves as a strong **initial predictive model**.


## Interpretation of Logistic Regression Model Coefficients

Logistic regression gives a set of coefficients, one for each feature variable, that show how much a variable affects the probability of diabetes (Outcome = 1). Coefficients can be interpreted in the **log-odds** space:

- **Positive coefficient** --> an increase in the value of the variable increases the likelihood of diabetes.
- **Negative coefficient** --> an increase in the value of the variable decreases the likelihood of diabetes.
- **Higher absolute value of the coefficient** --> indicates a stronger relationship between the variable and the outcome, assuming the variables are on comparable scales.

However, to make the interpretation clearer, we can transform the coefficients into **odds ratios (OR)**, which can be calculated using the formula:

$$
\text{Odds Ratio (OR)} = \exp(\beta)
$$

So:

- **OR > 1** --> higher values of the variable are associated with higher odds of diabetes.
- **OR < 1** --> higher values of the variable are associated with lower odds of diabetes.
- **OR = 1** --> the variable has no effect on the odds of diabetes.


In [8]:
odds_ratios = coefficients_to_odds_ratios(log_reg, X.columns)
odds_ratios

,Feature,Coefficient,Odds Ratio (exp(coeff))
0,Glucose,1.182511,3.262558
1,BMI,0.688735,1.991194
2,Pregnancies,0.377502,1.458637
3,DiabetesPedigreeFunction,0.233386,1.262869
4,Age,0.147798,1.159279
5,SkinThickness,0.028225,1.028627
6,BloodPressure,-0.044066,0.956891
7,Insulin,-0.066157,0.935983


### Key Insights

- **Glucose is the strongest risk factor**, with an odds ratio of approximately **3.26**, meaning that for each standardized unit increase in glucose level, the odds of being diabetic are **over three times higher**, holding other variables constant.

- **BMI is the second most influential feature**, nearly **doubling diabetes risk** (**OR ≈ 1.99**). This aligns with established medical research linking obesity to insulin resistance.

- **Pregnancies, family history (DPF), and age** also meaningfully increase diabetes risk, but not as strongly as glucose or BMI.

- **SkinThickness, BloodPressure, and Insulin** show weak predictive influence in this model. The small or negative coefficients may reflect:
  - Multicollinearity among predictors
  - Reduced influence after data imputation
  - Variability in individual physiological responses
  - Limited signal strength relative to glucose and BMI

- The negative coefficients for **BloodPressure** and **Insulin** do not necessarily imply that these features reduce diabetes risk clinically. Rather, within this dataset and model, they offer less predictive value after adjusting for stronger features.

### Conclusion

The logistic regression results are consistent with known medical literature: **elevated glucose and high BMI are the most impactful predictors of diabetes**. Pregnancy history, family genetics, and age contribute additional risk, while some physiological measures have weaker influence in this dataset.


## Save the trained model

In [9]:
save_artifacts(log_reg, scaler)
print("Saved to models/logistic_regression.joblib and models/scaler.joblib")

Saved to models/logistic_regression.joblib and models/scaler.joblib


For interactive predictions on new patient data, see `app/streamlit_app.py` or run `python main.py predict --help`.